# PriX (JAX) vs. Curl (PyTorch) — GPU Benchmark on Google Colab

Make sure **Runtime → Change runtime type → Hardware accelerator** is set to **T4 GPU** (or **L4 / A100 GPU**).

In [ ]:
!nvidia-smi
!pip install -q pywavelets pyyaml omegaconf onnx onnxruntime pandas scikit-learn scipy future cloudpickle
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [ ]:
# Clone or update PriX and Curl repositories, then install the CUDA build of JAX LAST:
# the Curl/CrypTen dependencies pull a CPU-only jaxlib, which would make the PriX
# subprocess fail with "Unknown backend: 'gpu' requested ... Platforms present are: cpu".
!if [ -d /content/prix ]; then git -C /content/prix pull; else git clone https://github.com/mugurbil/prix.git /content/prix; fi
!if [ -d /content/curl ]; then git -C /content/curl pull; else git clone https://github.com/jimouris/curl.git /content/curl; fi
!pip install -q -e /content/prix
!pip install -q -U "jax[cuda12]"
import jax
print("JAX", jax.__version__, "backend:", jax.default_backend(), jax.devices())
assert jax.default_backend() == "gpu", "JAX does not see the GPU: select a GPU runtime and re-run this cell"

In [ ]:
# Run PriX GPU LLM benchmarks (Wavelet LUT mode & Fission Evaluator mode)
!python /content/prix/examples/llms/launcher.py --world_size 2 --multiprocess --model BertTiny --tensor_size 1,16 --not-full --device cuda
!python /content/prix/examples/llms/launcher.py --world_size 2 --multiprocess --model GPT2 --tensor_size 1,16 --not-full --device cuda
!python /content/prix/examples/llms/launcher.py --world_size 2 --evaluator_size 1 --multiprocess --model GPT2 --tensor_size 1,16 --not-full --device cuda

In [ ]:
# Full side-by-side GPU benchmark: PriX (JAX) vs. CrypTen (PyTorch), 5 modes, both frameworks.
# PriX uses its default peer-to-peer TCP transport; CrypTen uses gloo over TCP. ~45-60 min on a T4.
!python3 -c "import jax; assert jax.default_backend() == 'gpu', jax.devices()"
!python3 /content/prix/benchmarks/benchmark_prix_vs_crypten.py --device cuda --comm-stats --repeats 5 --llm-repeats 5 \
    --prix-root /content/prix --crypten-root /content/curl --out /content/gpu_tesla_t4.json --overwrite
from google.colab import files
files.download("/content/gpu_tesla_t4.json")  # -> copy into benchmarks/results/gpu_tesla_t4.json

In [ ]:
# Optional: GPU transport comparison (PriX over gloo and over shared memory), appended to the same file.
!python3 /content/prix/benchmarks/benchmark_prix_vs_crypten.py --device cuda --comm-stats --framework prix --prix-transport gloo --prix-root /content/prix --crypten-root /content/curl --out /content/gpu_results.json
!python3 /content/prix/benchmarks/benchmark_prix_vs_crypten.py --device cuda --comm-stats --framework prix --prix-transport shm --prix-root /content/prix --crypten-root /content/curl --out /content/gpu_results.json
!python3 /content/prix/benchmarks/transport_ablation.py /content/gpu_results.json
from google.colab import files
files.download("/content/gpu_results.json")